In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr
import zarr

DATA_ROOT = Path.home() / "ml-ds_data" / "input_data"
INPUT_FILE = DATA_ROOT / "all_years.zarr"

In [ ]:
try:
    ds = xr.open_zarr(INPUT_FILE, consolidated=True)
except KeyError:
    print("Metadata not consolidated. Consolidating now...")
    zarr.consolidate_metadata(INPUT_FILE)
    ds = xr.open_zarr(INPUT_FILE, consolidated=True)

In [ ]:
ds

In [ ]:
nan_counts = ds.isnull().sum().compute()

In [ ]:
nan_counts

In [ ]:
ds["x_siconc"] = ds["x_siconc"].fillna(0)

In [ ]:
int(ds["x_siconc"].isnull().sum().compute().values)

In [ ]:
ds.chunks

In [ ]:
time_index = pd.Index(ds.time.values)
print(time_index.has_duplicates)

In [ ]:
mean = ds.mean(dim=("time", "y", "x"))
std  = ds.std(dim=("time", "y", "x"))

In [ ]:
mean = mean.rename({v: f"{v}_mean" for v in mean.data_vars})
std  = std.rename({v: f"{v}_std"  for v in std.data_vars})

stats = xr.merge([mean, std])

In [ ]:
stats.to_zarr(DATA_ROOT / "era5_normalization_stats.zarr", mode="w")

In [ ]:
from numcodecs import Blosc

output_path = DATA_ROOT / "data.zarr"

ds_to_save = ds.chunk({"time": 3})

compressor = Blosc(cname="zstd", clevel=3, shuffle=Blosc.SHUFFLE)
encoding = {
    var: {
        "compressor": compressor,
        "chunks": tuple(chunk_sizes[0] for chunk_sizes in ds_to_save[var].chunks),
    }
    for var in ds_to_save.data_vars
}

ds_to_save.to_zarr(output_path, mode="w", encoding=encoding, zarr_format=2, align_chunks=True)
print(f"Saved compressed dataset to {output_path}")